<a href="https://colab.research.google.com/github/lilydeller/Math344-Project/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part A


Import libraries for downloading and working with the data. Certificate verification is skipped to get around the download error and then the related warning messages are hidden. Then it downloads the dataset if it hasn't been downloaded before. Then the zip file is extracted. Then the ratings are put into a table with user, item, rating, and time columns. Then displays the first 5 rows and the number of ratings.

In [19]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import requests

In [20]:
ML_DATASET = Path("ml-100k")

if not ML_DATASET.exists():
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    response = requests.get(url, verify=False)

    ZipFile(BytesIO(response.content)).extractall()

In [21]:
movie_ratings = pd.read_csv(
    ML_DATASET / "u.data", sep="\t", names=["user", "item", "rating", "time"]
)

print(movie_ratings.head())
print("Number of ratings:", len(movie_ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


# Part B

A function that split the ratings into training data and testing data by randomly selecting ratings without replacement through using a permutation of the positions of the ratings. It takes in a percentage of ratings where it is probability of a rating to be selected as test data, and it returns array of Booleans where if the rating data at that position is testing data, then it is true, otherwise false. And the function also returns the testing data array inverted to create the positions of ratings that is training data. If a rating is selected as test data and the user and movie of that rating has at least two ratings in the training data, then mark the rating at that position as test data and decrement the amount of ratings that the user and movie has for the training data. It also takes a seed for the random number generator to create the permutation and select test data. Then split the movie ratings into training data and testing data using the default test fraction of 0.10 and seed 0.

In [26]:
def split_ratings(ratings: pd.DataFrame, test_fraction=0.10, seed=0):
    # Hide ratings without removing a user's or movie's last training rating.
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=np.bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if rng.random() < test_fraction and user_left[user] > 1 and item_left[item] > 1:
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

In [25]:
train, test = split_ratings(movie_ratings)
print("train: ", len(train), "test: ", len(test))
print("test fraction: ", round(len(test) / len(movie_ratings), 3))

train:  90032 test:  9968
test fraction:  0.1


# Part C

# Part D

# Part E

# Part F

# Practice Exercises

## Exercise 1

## Exercise 2

## Exercise 3